# VoxBridge - Notebook 01: Model Build Pipeline

## Run this notebook only when model artifacts change

Examples: a selected ASR export change, an ONNX export or quantization change, an OPUS-MT model/tokenizer
change, or a Piper voice change. Normal Kotlin/runtime/test changes
use `00_dev_bootstrap.ipynb` and its **UPDATE SOURCE + TEST** cell instead.

This notebook works in `/content/voxbridge/models`, creates PhoWhisper Base + Whisper Small with
the shared VAD/MT/TTS artifacts, runs the real JVM E2E gate once, then atomically promotes the
validated cache to `MyDrive/VoxBridge/models/`. A failed/incomplete
export cannot overwrite the known-good cache. A timestamped archive is created afterward.


## 0. Mount Drive and clone or update the private source

This notebook is independent: it can run in a fresh Colab runtime without Notebook 00.


In [ ]:
import os
import pathlib
import subprocess
from google.colab import drive, userdata

DRIVE_ROOT = "/content/drive/MyDrive/VoxBridge"
REPO_URL = "https://github.com/tanphong-sudo/voxbridge.git"
REPO = pathlib.Path("/content/voxbridge")
drive.mount("/content/drive")


def run_git(args, cwd=None):
    """Run Git with the token held only in this process environment."""
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Missing GITHUB_TOKEN. In the left sidebar, open Secrets, add GITHUB_TOKEN, "
            "and enable its notebook access. The token needs read access to voxbridge."
        )

    askpass = pathlib.Path("/tmp/voxbridge-git-askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) echo x-access-token ;;\n"
        "  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\n"
        "esac\n"
    )
    askpass.chmod(0o700)
    env = {
        **os.environ,
        "GITHUB_TOKEN": token,
        "GIT_ASKPASS": str(askpass),
        "GIT_TERMINAL_PROMPT": "0",
    }
    try:
        return subprocess.run(["git", *args], cwd=cwd, env=env, text=True, check=True)
    finally:
        askpass.unlink(missing_ok=True)


if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(
            f"{REPO} exists but is not a Git checkout. Run: !rm -rf /content/voxbridge, then rerun this cell."
        )
    run_git(["clone", "--branch", "main", REPO_URL, str(REPO)])
else:
    run_git(["fetch", "origin", "main"], cwd=REPO)
    run_git(["pull", "--ff-only", "origin", "main"], cwd=REPO)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
print(f"Source ready: {REPO} @ {commit}")


In [ ]:
# Pin everything the CPU-side exports need (verified combination). Isolated venv: Colab's own
# site-packages stay untouched.
import subprocess, sys
VENV_TOOL = "/content/venvs/tooling"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"], check=True)
subprocess.run([sys.executable, "-m", "uv", "venv", "--clear", "--python", "3.11", "--seed", VENV_TOOL], check=True)
PIP = f"{VENV_TOOL}/bin/pip"
subprocess.run([PIP, "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([PIP, "install", "-q", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([PIP, "install", "-q",
    "numpy==1.26.4",                       # CPU torch (small wheel, enough for export)
    "transformers==4.42.4", "optimum[onnxruntime]==1.21.0",
    "onnx==1.17.0", "onnxruntime==1.19.2",
    "sentencepiece==0.2.0", "sacremoses==0.1.1",
    "huggingface_hub==0.26.2", "librosa==0.10.2",
    "sherpa-onnx==1.13.3",                                 # render_safety_cache uses the SAME engine version as the app's AAR
], check=True)
print("tooling venv ready")
subprocess.run([PIP, "check"], check=True)


## 3 · OPUS-MT — all 6 hops (ONNX + INT8) · 4 · VAD + Piper ×4 voices · 5 · safety-phrase cache

Contest direction set (OneVoice AI Challenge: VI↔EN, VI↔Mandarin, VI↔Korean). VI→ZH / VI→KO /
KO→VI have no direct Helsinki graph, so the app chains through EN (`ChainedTranslator`) — the
six hops exported here are exactly: vi-en, en-vi, en-zh, zh-vi (direct ZH→VI), en-ko (tc-big), ko-en.


In [ ]:
# fetch_all.sh creates the selected ASRs plus OPUS/VAD/Piper/cache with the venv Python.
e2e_marker = pathlib.Path("/content/voxbridge/.colab-model-e2e-ok")
e2e_marker.unlink(missing_ok=True)
env = {"PYTHON": "/content/venvs/tooling/bin/python", "PATH": "/content/venvs/tooling/bin:/usr/bin:/bin"}
r = subprocess.run(["bash", "model-tools-python/scripts/fetch_all.sh"],
                   cwd="/content/voxbridge", env={**os.environ, **env})
assert r.returncode == 0, "fetch_all.sh failed - read the log above"
!du -sh /content/voxbridge/models/*/

## 6 · Verify the graphs (the ⚠️ rows of MODEL_MANIFEST, now answered for real)

In [ ]:
import subprocess
subprocess.run(["/content/venvs/tooling/bin/python", "-c", '# Load every exported ONNX and print its REAL tensor I/O - this is exactly what the Android\n# runners introspect at runtime; nothing in the app hard-codes these names.\nimport onnx, pathlib, json\nMODELS = pathlib.Path("/content/voxbridge/models")\nTRANSLATION_DIRS = [\n    "translation/opus-mt-vi-en", "translation/opus-mt-en-vi",\n    "translation/opus-mt-en-zh", "translation/opus-mt-zh-vi",\n    "translation/opus-mt-en-ko", "translation/opus-mt-ko-en",\n]\ndef show(p):\n    m = onnx.load(str(p), load_external_data=False)\n    ins = [i.name for i in m.graph.input]; outs = [o.name for o in m.graph.output]\n    print(f"  {p.name}:")\n    print(f"    inputs : {ins[:6]}{\' ...\' if len(ins) > 6 else \'\'} ({len(ins)})")\n    print(f"    outputs: {outs[:6]}{\' ...\' if len(outs) > 6 else \'\'} ({len(outs)})")\n    return {"file": p.name, "inputs": ins, "outputs": outs}\nreport = {}\nfor d in TRANSLATION_DIRS + ["vad"]:\n    print(f"[{d}]")\n    for f in sorted((MODELS / d).glob("*.onnx")):\n        report[str(f.relative_to(MODELS))] = show(f)\n# sanity: the optimum KV contract the Kotlin decode loops rely on (checked on the vi-en graph;\n# every hop comes from the same optimum OnnxConfigWithPast machinery, so the contract is shared)\nmt = report.get("translation/opus-mt-vi-en/decoder_with_past_model.onnx")\nassert mt is not None, "decoder_with_past_model.onnx khong ton tai trong export - optimum khong xuat with-past graph, chay lai cell 10."\nassert any(n.startswith("past_key_values.") for n in mt["inputs"]), "with-past graph lacks past_key_values inputs"\nassert any(n.startswith("present.") for n in mt["outputs"]), "with-past graph lacks present outputs"\npf = report.get("translation/opus-mt-vi-en/decoder_model.onnx")\nassert pf is not None, "decoder_model.onnx khong ton tai trong export - optimum khong xuat past-free graph, chay lai cell 10."\nassert any(n.startswith("present.") for n in pf["outputs"]), "past-free graph must emit present.* (KV bootstrap)"\n# EVERY hop dir must carry the full triple the app gate requires: an encoder graph + a decoder\n# graph (quantized or plain) + the 3 tokenizer files. The old encoder-only assert let\n# "6/6 exported" print for a hop with just an encoder (Round-4 review LOW) — the notebook-02\n# gate would still catch it, but late; fail here instead.\ndef hop_missing(d):\n    miss = []\n    have = lambda stem: (MODELS / d / f"{stem}.onnx").exists() or (MODELS / d / f"{stem}_quantized.onnx").exists()\n    if not have("encoder_model"):\n        miss.append("encoder_model[_quantized].onnx")\n    if not (have("decoder_model") or have("decoder_with_past_model")):\n        miss.append("decoder_model[_quantized].onnx")\n    for tok in ("source.spm", "target.spm", "vocab.json"):\n        if not (MODELS / d / tok).exists():\n            miss.append(tok)\n    return miss\nfor d in TRANSLATION_DIRS:\n    miss = hop_missing(d)\n    assert not miss, f"{d} thieu: {\'; \'.join(miss)} - export_opus_mt.py chua hoan thanh cho hop nay, chay lai cell 10."\nprint("\\nKV contract OK: past-free emits present.*, with-past consumes past_key_values.* / emits decoder part only.")\nprint(f"OK: {len(TRANSLATION_DIRS)}/6 MT hops exported (encoder + decoder + tokenizer, gate-grade).")\njson.dump(report, open("/content/voxbridge/models/io_report.json", "w"), indent=2)\n'], check=True)


## 7 · E2E JVM — chạy decode Kotlin thật với model vừa export
Cell này compile **đúng code Kotlin của app** (`core/` + `runtime/`) bằng kotlinc và chạy
`tools/e2e-jvm/run.sh` với ONNX Runtime desktop 1.20.0 (cùng major-minor với
`onnxruntime-android-qnn` trong app) trên **đúng các file model vừa export**. Bắt lệch
contract **trước khi** build APK — notebook 02 không phải dò lỗi của notebook 01.

Assert (trung thực, không tổng hợp speech giả):
- **VAD**: 30 s silence → speech prob < 0.5
- **Whisper**: `WhisperAsr.load` + decode vòng KV thật trên 30 s silence → transcript rỗng + `noSpeechProbability > 0.5`
- **OPUS-MT**: cả 2 chiều dịch `xin chào` / `hello world` → output khác rỗng + `translationScore ∈ (0,1]`

Thiếu PhoWhisper Base / Whisper Small hoặc contract lệch → **FAIL**; chỉ khi E2E xanh mới được promote cache sang Drive.

In [ ]:
# JVM e2e: compile real Kotlin runtime+core, run Whisper/VAD/OPUS-MT decode on the
# freshly exported graphs. Missing selected models or contract drift FAIL loudly.
import os, shutil, subprocess

JAVA_HOME_17 = '/usr/lib/jvm/java-17-openjdk-amd64'
if not os.path.isdir(JAVA_HOME_17):
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'openjdk-17-jdk-headless'], check=True)
os.environ['JAVA_HOME'] = JAVA_HOME_17
os.environ['PATH'] = JAVA_HOME_17 + '/bin:' + os.environ['PATH']

r = subprocess.run(
    ['bash', 'tools/e2e-jvm/run.sh', 'models'],
    cwd='/content/voxbridge',
    env={**os.environ, 'HOME': os.environ.get('HOME', '/root')},
)
assert r.returncode == 0, 'e2e-jvm FAILED — fix the export/contract above before building the APK'

e2e_marker = pathlib.Path("/content/voxbridge/.colab-model-e2e-ok")
e2e_marker.write_text("ok\n")
print("E2E gate passed; promotion marker written:", e2e_marker)


In [ ]:
# Promote only a full, E2E-validated model tree to Drive, then archive it.
import pathlib, shutil, subprocess, time
from google.colab import drive

drive.mount("/content/drive")
repo = pathlib.Path("/content/voxbridge")
models = repo / "models"
assert models.is_dir(), "Missing models/; the export cells did not complete."
e2e_marker = repo / ".colab-model-e2e-ok"
assert e2e_marker.is_file() and e2e_marker.read_text().strip() == "ok", "E2E marker missing; run the E2E cell before promotion."
subprocess.run(["bash", "tools/dev/sync-models.sh", "models", "/content/drive/MyDrive/VoxBridge/models"], cwd=repo, check=True)

tag = time.strftime("%Y%m%d-%H%M")
artifacts = pathlib.Path("/content/drive/MyDrive/VoxBridge/artifacts/models")
artifacts.mkdir(parents=True, exist_ok=True)
out = pathlib.Path(f"/content/voxbridge-models-{tag}.zip")
tmp = out.with_suffix(".zip.part")
if tmp.exists(): tmp.unlink()
subprocess.run(["zip", "-qr", str(tmp), "models"], cwd=repo, check=True)
tmp.replace(out)
shutil.copy2(out, artifacts / out.name)
print("Drive cache:", "/content/drive/MyDrive/VoxBridge/models")
print("Model archive:", artifacts / out.name)
